In [ ]:
import evaluate
import pandas as pd

rouge = evaluate.load("rouge")
bert_score = evaluate.load("bertscore")
gt = pd.read_json('datasets/xsum/sorted_test_gpt4_turbo.json')


In [ ]:
pred_path = 'datasets/xsum/xsum_human_generated_predictions.json'
pred = pd.read_json(pred_path)

In [ ]:
# Compute per-sample ROUGE and BERTScore in one batch
import json
import time

# Prepare the data
predictions = pred['summary'].tolist()
references = gt['summary'].tolist()

print(f"{len(predictions)} samples in total")
start_time = time.time()

# Compute BERTScore for every sample in a single call
print("Computing BERTScore...")
bert_start = time.time()
bert_results = bert_score.compute(predictions=predictions, references=references, lang="en", batch_size=64)
print(f"BERTScore done in {time.time() - bert_start:.2f}s")

# Batch-compute all ROUGE scores (use_aggregator=False returns per-sample scores)
print("Computing ROUGE...")
rouge_start = time.time()
rouge_results = rouge.compute(
    predictions=predictions,
    references=references,
    use_aggregator=False,  # key: return per-sample scores instead of the mean
    use_stemmer=True
)
print(f"ROUGE done in {time.time() - rouge_start:.2f}s")

# Assemble the results (list comprehension is faster than repeated append)
print("Assembling results...")
results = [
    {
        'id': idx,
        'prediction': predictions[idx],
        'reference': references[idx],
        'rouge1': rouge_results['rouge1'][idx],
        'rouge2': rouge_results['rouge2'][idx],
        'rougeL': rouge_results['rougeL'][idx],
        'rougeLsum': rouge_results['rougeLsum'][idx],
        'bertscore_precision': float(bert_results['precision'][idx]),
        'bertscore_recall': float(bert_results['recall'][idx]),
        'bertscore_f1': float(bert_results['f1'][idx])
    }
    for idx in range(len(predictions))
]

# Write to a JSON file
output_path = pred_path.replace('generated_predictions.json', 'evaluation_results_gpt.json')
print(f"Writing results to {output_path} ...")
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

total_time = time.time() - start_time
print(f"\nDone. Total time: {total_time:.2f}s")
print(f"Saved scores for {len(results)} samples to: {output_path}")
print(f"Average per sample: {total_time/len(results)*1000:.2f} ms")

# Summary statistics
print("\n" + "="*60)
print("Summary statistics")
print("="*60)

import numpy as np

# ROUGE statistics
print("\n[ROUGE scores]")
print(f"  ROUGE-1: {np.mean(rouge_results['rouge1']):.4f} (±{np.std(rouge_results['rouge1']):.4f})")
print(f"  ROUGE-2: {np.mean(rouge_results['rouge2']):.4f} (±{np.std(rouge_results['rouge2']):.4f})")
print(f"  ROUGE-L: {np.mean(rouge_results['rougeL']):.4f} (±{np.std(rouge_results['rougeL']):.4f})")
print(f"  ROUGE-Lsum: {np.mean(rouge_results['rougeLsum']):.4f} (±{np.std(rouge_results['rougeLsum']):.4f})")

# BERTScore statistics
print("\n【BERTScore】")
print(f"  Precision: {np.mean(bert_results['precision']):.4f} (±{np.std(bert_results['precision']):.4f})")
print(f"  Recall:    {np.mean(bert_results['recall']):.4f} (±{np.std(bert_results['recall']):.4f})")
print(f"  F1:        {np.mean(bert_results['f1']):.4f} (±{np.std(bert_results['f1']):.4f})")

print("\n" + "="*60)